# AERONET as Atmospheric Context for Instrument Agreement

Uses AERONET as an independent atmospheric-condition monitor to explain **when and why** optical methods (HIPS, aethalometer) disagree with thermal EC (FTIR).

## Scope (stays in instrument-methodology, not source apportionment)
- AERONET provides daytime, column-integrated optical information
- We use it to test whether MAC variability, HIPS baseline offset, and compression are linked to measurable atmospheric conditions
- All framing is "measurement interference" and "method sensitivity," not sources

## Structure
1. **Dual MAC: HIPS vs Aethalometer** — MAC_HIPS and MAC_AETH vs PW (tercile boxplots + multivariate regression)
2. **HIPS Residual vs Coarse-Mode Regime** — residual from best-fit HIPS–EC line vs τc, FMF, τc/τa
3. **Compression by AOD Terciles** — HIPS–EC slope/intercept split by total AOD loading
4. **Daytime (10–14h) vs 24h Representativeness** — does restricting to AERONET hours improve agreement?
5. **Synthesis** — three paper-ready plots

---

## Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

RESEARCH_DIR = '/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem'
scripts_path = os.path.join(RESEARCH_DIR, 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, FILTER_DATA_PATH, PROCESSED_SITES_DIR, MAC_VALUE
from data_matching import (
    load_aethalometer_data, load_filter_data, match_all_parameters
)

SEASONS = {'Dry': [10, 11, 12, 1, 2], 'Belg': [3, 4, 5], 'Kiremt': [6, 7, 8, 9]}
SEASON_COLORS = {'Dry': '#E67E22', 'Belg': '#27AE60', 'Kiremt': '#3498DB'}
SEASON_ORDER = ['Dry', 'Belg', 'Kiremt']

def assign_season(month):
    for name, months in SEASONS.items():
        if month in months:
            return name
    return 'Unknown'

AERONET_MISSING = -999.
SITE_CODE = 'ETAD'

# MA350 manufacturer specific attenuation cross-section at 880nm
SIGMA_AETH_IR = 7.77  # m²/g — used to back-calculate babs from reported BCc

print(f'HIPS MAC (config): {MAC_VALUE} m²/g')
print(f'Aeth σ_ATN at 880nm: {SIGMA_AETH_IR} m²/g')
print('Setup complete')

## Data Loading

In [ ]:
# === Paths ===
BC_MINUTE_PATH = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'Aethelometry Data/JacrosMA350 60s Data20250804082112/'
    'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl'
)
AERONET_BASE = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'AERONET/Jacros'
)
AERONET_AOD_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.lev20')
AERONET_SDA_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20')
AERONET_AOD_ALLPTS = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.lev20')
AERONET_SDA_ALLPTS = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20')

print('Paths configured')

In [ ]:
# === Load minute-level BC (local time, Addis = UTC+3) ===
bc_raw = pd.read_pickle(BC_MINUTE_PATH)
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)

for col in ['UV BCc', 'IR BCc', 'UV BC1', 'IR BC1']:
    if col in bc_raw.columns:
        bc_raw[col] = bc_raw[col] / 1000  # ng/m³ → µg/m³

for col in ['UV BCc', 'IR BCc']:
    if col in bc_raw.columns:
        bc_raw.loc[bc_raw[col] < 0, col] = np.nan
        mu, sigma = bc_raw[col].mean(), bc_raw[col].std()
        bc_raw.loc[bc_raw[col] > mu + 3 * sigma, col] = np.nan

print(f'BC minute data: {len(bc_raw):,} records')
print(f'  Range: {bc_raw.index.min()} → {bc_raw.index.max()}')

In [ ]:
# === Load AERONET ===
def load_aeronet_daily(filepath, date_col='Date(dd:mm:yyyy)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['Date'] = pd.to_datetime(df[date_col], format='%d:%m:%Y')
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

def load_aeronet_allpoints(filepath, date_col='Date(dd:mm:yyyy)', time_col='Time(hh:mm:ss)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['datetime_utc'] = pd.to_datetime(
        df[date_col] + ' ' + df[time_col], format='%d:%m:%Y %H:%M:%S')
    df.set_index('datetime_utc', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_daily = load_aeronet_daily(AERONET_AOD_DAILY)
sda_daily = load_aeronet_daily(AERONET_SDA_DAILY, date_col='Date_(dd:mm:yyyy)')

aod_all = load_aeronet_allpoints(AERONET_AOD_ALLPTS)
sda_all = load_aeronet_allpoints(AERONET_SDA_ALLPTS,
                                  date_col='Date_(dd:mm:yyyy)', time_col='Time_(hh:mm:ss)')

for df in [aod_daily, sda_daily]:
    df['month'] = df.index.month
    df['season'] = df['month'].map(assign_season)

# Column aliases
COL_TC = 'Coarse_Mode_AOD_500nm[tau_c]'
COL_TF = 'Fine_Mode_AOD_500nm[tau_f]'
COL_FMF = 'FineModeFraction_500nm[eta]'
COL_PW = 'Precipitable_Water(cm)'
COL_AOD = 'AOD_500nm'
COL_AE = '440-870_Angstrom_Exponent'

print(f'AERONET daily AOD: {len(aod_daily)} days, SDA: {len(sda_daily)} days')
print(f'AERONET all-points AOD: {len(aod_all):,}, SDA: {len(sda_all):,}')

In [ ]:
# === Load 24h filter data and compute MAC values ===
aeth_data = load_aethalometer_data()
filter_data = load_filter_data()

df_matched = match_all_parameters('Addis_Ababa', SITE_CODE,
                                   aeth_data['Addis_Ababa'], filter_data)

df_matched['date'] = pd.to_datetime(df_matched['date'])
df_matched['month'] = df_matched['date'].dt.month
df_matched['season'] = df_matched['month'].map(assign_season)

# --- Compute MAC values ---
# hips_fabs from match_all_parameters = HIPS_Fabs / MAC_VALUE (µg/m³ EC equivalent)
# ir_bcc = 24h-avg aethalometer BC (µg/m³)
# ftir_ec = FTIR thermal EC (µg/m³)

valid = df_matched.dropna(subset=['ftir_ec']).copy()
valid = valid[valid['ftir_ec'] > 0]

# MAC_HIPS = HIPS_Fabs / FTIR_EC = (hips_fabs * MAC_VALUE) / ftir_ec
hips_mask = valid['hips_fabs'].notna()
valid.loc[hips_mask, 'mac_hips'] = (
    valid.loc[hips_mask, 'hips_fabs'] * MAC_VALUE) / valid.loc[hips_mask, 'ftir_ec']

# MAC_AETH = babs_aeth / FTIR_EC = (ir_bcc * SIGMA_AETH) / ftir_ec
aeth_mask = valid['ir_bcc'].notna()
valid.loc[aeth_mask, 'mac_aeth'] = (
    valid.loc[aeth_mask, 'ir_bcc'] * SIGMA_AETH_IR) / valid.loc[aeth_mask, 'ftir_ec']

# Also compute the simple ratios (mass space)
valid.loc[hips_mask, 'hips_ftir_ratio'] = valid.loc[hips_mask, 'hips_fabs'] / valid.loc[hips_mask, 'ftir_ec']
valid.loc[aeth_mask, 'aeth_ftir_ratio'] = valid.loc[aeth_mask, 'ir_bcc'] / valid.loc[aeth_mask, 'ftir_ec']

print(f'Matched filter samples with FTIR EC > 0: {len(valid)}')
for col in ['mac_hips', 'mac_aeth']:
    n = valid[col].notna().sum()
    if n > 0:
        print(f'  {col}: n={n}, mean={valid[col].mean():.1f}, median={valid[col].median():.1f} m²/g')

In [ ]:
# === Merge filter data with daily AERONET ===
aeronet_daily_combined = aod_daily[
    [COL_AOD, COL_PW, COL_AE]
].join(sda_daily[[COL_TC, COL_TF, COL_FMF]], how='outer')

# Compute coarse fraction
aeronet_daily_combined['coarse_fraction'] = (
    aeronet_daily_combined[COL_TC] / aeronet_daily_combined[COL_AOD])

filt = valid.copy()
filt['date_key'] = filt['date'].dt.normalize()
aeronet_daily_combined.index = pd.to_datetime(aeronet_daily_combined.index)

filt = filt.merge(aeronet_daily_combined, left_on='date_key',
                  right_index=True, how='left')

n_aod = filt[COL_AOD].notna().sum()
n_pw = filt[COL_PW].notna().sum()
n_tc = filt[COL_TC].notna().sum()
print(f'Filter samples with AERONET: AOD={n_aod}, PW={n_pw}, τc={n_tc} / {len(filt)}')

---

## 1. Dual MAC: HIPS vs Aethalometer

Compute MAC for both optical methods:
- **MAC_HIPS** = babs_HIPS / EC = (HIPS_Fabs) / (FTIR EC)
- **MAC_AETH** = babs_AETH / EC = (BC_aeth × σ_aeth) / (FTIR EC)

If PW affects HIPS (filter-based, 24h integrated) more than the aethalometer (real-time, with loading correction), that isolates the mechanism to **filter-based** optical absorption specifically.

In [ ]:
# --- Dual MAC scatter vs PW ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (mac_col, mac_label, expected) in enumerate([
    ('mac_hips', f'MAC$_{{HIPS}}$ (m²/g)', MAC_VALUE),
    ('mac_aeth', f'MAC$_{{AETH}}$ (m²/g)', SIGMA_AETH_IR),
]):
    ax = axes[i]
    v = filt.dropna(subset=[mac_col, COL_PW]).copy()
    # Clip extreme outliers
    q01, q99 = v[mac_col].quantile(0.01), v[mac_col].quantile(0.99)
    v = v[(v[mac_col] >= q01) & (v[mac_col] <= q99)]
    
    for season in SEASON_ORDER:
        s = v[v['season'] == season]
        ax.scatter(s[COL_PW], s[mac_col], color=SEASON_COLORS[season],
                  alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
    
    r, p = stats.pearsonr(v[COL_PW], v[mac_col])
    sl, ic, _, _, _ = stats.linregress(v[COL_PW], v[mac_col])
    x_fit = np.linspace(v[COL_PW].min(), v[COL_PW].max(), 100)
    ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2.5)
    ax.axhline(y=expected, color='red', linestyle='--', alpha=0.6,
               label=f'Default = {expected}')
    
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}\nn = {len(v)}',
            transform=ax.transAxes, fontsize=11, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel('Precipitable Water (cm)', fontsize=12)
    ax.set_ylabel(mac_label, fontsize=12)
    ax.set_title(f'{mac_label.split("(")[0].strip()} vs Column Moisture',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Panel 3: Side-by-side comparison
ax = axes[2]
both = filt.dropna(subset=['mac_hips', 'mac_aeth', COL_PW]).copy()
q01_h, q99_h = both['mac_hips'].quantile(0.01), both['mac_hips'].quantile(0.99)
q01_a, q99_a = both['mac_aeth'].quantile(0.01), both['mac_aeth'].quantile(0.99)
both = both[(both['mac_hips'] >= q01_h) & (both['mac_hips'] <= q99_h) &
            (both['mac_aeth'] >= q01_a) & (both['mac_aeth'] <= q99_a)]

r_hips, p_hips = stats.pearsonr(both[COL_PW], both['mac_hips'])
r_aeth, p_aeth = stats.pearsonr(both[COL_PW], both['mac_aeth'])

ax.scatter(both[COL_PW], both['mac_hips'], alpha=0.4, s=40,
           color='#E74C3C', label=f'MAC_HIPS (r={r_hips:.3f})', marker='o')
ax.scatter(both[COL_PW], both['mac_aeth'], alpha=0.4, s=40,
           color='#3498DB', label=f'MAC_AETH (r={r_aeth:.3f})', marker='^')

for mac_col, color in [('mac_hips', '#E74C3C'), ('mac_aeth', '#3498DB')]:
    sl, ic, _, _, _ = stats.linregress(both[COL_PW], both[mac_col])
    ax.plot(x_fit, sl * x_fit + ic, '-', color=color, linewidth=2.5)

ax.set_xlabel('Precipitable Water (cm)', fontsize=12)
ax.set_ylabel('Effective MAC (m²/g)', fontsize=12)
ax.set_title('HIPS vs Aeth: Which Is More Sensitive to PW?',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Dual MAC Analysis: Precipitable Water Sensitivity',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nPW sensitivity comparison (matched samples, n={len(both)}):')
print(f'  MAC_HIPS vs PW: r = {r_hips:.3f}')
print(f'  MAC_AETH vs PW: r = {r_aeth:.3f}')
print(f'  → HIPS is {"more" if abs(r_hips) > abs(r_aeth) else "less"} '
      f'sensitive to PW than the aethalometer')

In [ ]:
# --- MAC in PW terciles (boxplots) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (mac_col, mac_label, expected) in enumerate([
    ('mac_hips', 'MAC$_{HIPS}$', MAC_VALUE),
    ('mac_aeth', 'MAC$_{AETH}$', SIGMA_AETH_IR),
]):
    ax = axes[i]
    v = filt.dropna(subset=[mac_col, COL_PW]).copy()
    q01, q99 = v[mac_col].quantile(0.01), v[mac_col].quantile(0.99)
    v = v[(v[mac_col] >= q01) & (v[mac_col] <= q99)]
    
    pw_33, pw_67 = v[COL_PW].quantile(0.33), v[COL_PW].quantile(0.67)
    v['pw_tercile'] = pd.cut(v[COL_PW], bins=[-np.inf, pw_33, pw_67, np.inf],
                              labels=[f'Low\n(<{pw_33:.1f} cm)',
                                      f'Mid\n({pw_33:.1f}-{pw_67:.1f} cm)',
                                      f'High\n(>{pw_67:.1f} cm)'])
    
    tercile_labels = v['pw_tercile'].cat.categories.tolist()
    plot_data = [v[v['pw_tercile'] == t][mac_col].dropna() for t in tercile_labels]
    
    bp = ax.boxplot(plot_data, labels=tercile_labels, patch_artist=True,
                   showfliers=False, widths=0.6)
    tercile_colors = ['#E67E22', '#27AE60', '#3498DB']
    for j, patch in enumerate(bp['boxes']):
        patch.set_facecolor(tercile_colors[j])
        patch.set_alpha(0.7)
        # Overlay points
        data = plot_data[j]
        x_jitter = np.random.normal(j + 1, 0.08, len(data))
        ax.scatter(x_jitter, data, color=tercile_colors[j],
                  alpha=0.3, s=15, edgecolors='k', linewidth=0.2)
        ax.text(j + 1, data.max() * 1.02,
                f'n={len(data)}\nmed={data.median():.1f}',
                ha='center', fontsize=9, va='bottom')
    
    ax.axhline(y=expected, color='red', linestyle='--', alpha=0.7,
               label=f'Default = {expected}')
    ax.set_xlabel('Precipitable Water Tercile', fontsize=12)
    ax.set_ylabel(f'{mac_label} (m²/g)', fontsize=12)
    ax.set_title(f'{mac_label} by PW Tercile', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Kruskal-Wallis test
    if all(len(d) > 3 for d in plot_data):
        h_stat, kw_p = stats.kruskal(*plot_data)
        ax.text(0.95, 0.05, f'Kruskal-Wallis\nH={h_stat:.1f}, p={kw_p:.2e}',
                transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.suptitle('MAC in Precipitable Water Terciles', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Multivariate regression: MAC ~ PW + season ---
for mac_col, mac_label in [('mac_hips', 'MAC_HIPS'), ('mac_aeth', 'MAC_AETH')]:
    v = filt.dropna(subset=[mac_col, COL_PW]).copy()
    q01, q99 = v[mac_col].quantile(0.01), v[mac_col].quantile(0.99)
    v = v[(v[mac_col] >= q01) & (v[mac_col] <= q99)]
    
    v['pw'] = v[COL_PW]
    v['mac'] = v[mac_col]
    
    # Model 1: MAC ~ PW only
    model_pw = ols('mac ~ pw', data=v).fit()
    
    # Model 2: MAC ~ PW + season
    model_pw_season = ols('mac ~ pw + C(season)', data=v).fit()
    
    # Model 3: MAC ~ PW + season + PW:season (interaction)
    model_interaction = ols('mac ~ pw * C(season)', data=v).fit()
    
    print(f'\n{"=" * 70}')
    print(f'{mac_label} Regression Models (n={len(v)})')
    print(f'{"=" * 70}')
    print(f'\nModel 1: {mac_label} ~ PW')
    print(f'  R² = {model_pw.rsquared:.3f}, Adj-R² = {model_pw.rsquared_adj:.3f}')
    print(f'  PW coeff = {model_pw.params["pw"]:.2f} (p={model_pw.pvalues["pw"]:.2e})')
    
    print(f'\nModel 2: {mac_label} ~ PW + season')
    print(f'  R² = {model_pw_season.rsquared:.3f}, Adj-R² = {model_pw_season.rsquared_adj:.3f}')
    print(f'  PW coeff = {model_pw_season.params["pw"]:.2f} (p={model_pw_season.pvalues["pw"]:.2e})')
    for key in model_pw_season.params.index:
        if 'season' in key:
            print(f'  {key}: {model_pw_season.params[key]:.2f} (p={model_pw_season.pvalues[key]:.3f})')
    
    print(f'\nModel 3: {mac_label} ~ PW × season (interaction)')
    print(f'  R² = {model_interaction.rsquared:.3f}, Adj-R² = {model_interaction.rsquared_adj:.3f}')
    
    # Does adding season improve over PW alone?
    from scipy.stats import f as f_dist
    r2_reduced = model_pw.rsquared
    r2_full = model_pw_season.rsquared
    df_extra = model_pw_season.df_model - model_pw.df_model
    df_resid = model_pw_season.df_resid
    if df_extra > 0:
        f_stat = ((r2_full - r2_reduced) / df_extra) / ((1 - r2_full) / df_resid)
        f_p = 1 - f_dist.cdf(f_stat, df_extra, df_resid)
        print(f'\n  Season adds over PW alone: F={f_stat:.2f}, p={f_p:.4f}')
        if f_p < 0.05:
            print(f'  → Season significantly improves fit beyond PW')
        else:
            print(f'  → PW alone captures most of the seasonal variation')

---

## 2. HIPS Residual vs Coarse-Mode Regime

Instead of raw HIPS/FTIR ratio, compute the **residual** from the best-fit HIPS–EC regression line:

$$\text{HIPS residual} = \text{babs}_{\text{HIPS}} - (\beta_0 + \beta_1 \cdot \text{EC})$$

Then test whether this residual correlates with coarse-mode optical regime indicators:
- τc (coarse-mode AOD) — direct coarse loading
- η (FMF) — fine-mode fraction  
- τc/τa (coarse fraction) — relative coarse importance

If HIPS residual increases on high-τc days → "coarse/dusty optical regime pushes HIPS off baseline."

In [ ]:
# Fit best HIPS–EC regression and compute residuals
hips_ec_valid = filt.dropna(subset=['hips_fabs', 'ftir_ec']).copy()
hips_ec_valid = hips_ec_valid[hips_ec_valid['ftir_ec'] > 0]

# Work in absorption space: HIPS_Fabs vs FTIR_EC
hips_ec_valid['hips_fabs_raw'] = hips_ec_valid['hips_fabs'] * MAC_VALUE

slope_fit, intercept_fit, r_fit, p_fit, se_fit = stats.linregress(
    hips_ec_valid['ftir_ec'], hips_ec_valid['hips_fabs_raw'])

hips_ec_valid['hips_predicted'] = slope_fit * hips_ec_valid['ftir_ec'] + intercept_fit
hips_ec_valid['hips_residual'] = hips_ec_valid['hips_fabs_raw'] - hips_ec_valid['hips_predicted']

# Transfer back to main dataframe
filt.loc[hips_ec_valid.index, 'hips_residual'] = hips_ec_valid['hips_residual']

print(f'Best-fit: HIPS_Fabs = {slope_fit:.2f} × EC + {intercept_fit:.3f}')
print(f'  r = {r_fit:.3f}, R² = {r_fit**2:.3f}')
print(f'  Slope = effective bulk MAC: {slope_fit:.1f} m²/g')
print(f'  Intercept = baseline offset: {intercept_fit:.3f}')
print(f'  Residuals: mean={hips_ec_valid["hips_residual"].mean():.4f}, '
      f'std={hips_ec_valid["hips_residual"].std():.3f}')

In [ ]:
# HIPS residual vs coarse-mode indicators
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

test_cols = [
    (COL_TC, 'Coarse-Mode AOD (τ$_c$)', 'coarse loading'),
    (COL_FMF, 'Fine-Mode Fraction (η)', 'fine dominance'),
    ('coarse_fraction', 'Coarse Fraction (τ$_c$/τ$_a$)', 'relative coarseness'),
    (COL_PW, 'Precipitable Water (cm)', 'column moisture'),
]

for i, (col, xlabel, description) in enumerate(test_cols):
    ax = axes[i]
    v = filt.dropna(subset=['hips_residual', col])
    if len(v) < 5:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue
    
    for season in SEASON_ORDER:
        s = v[v['season'] == season]
        ax.scatter(s[col], s['hips_residual'], color=SEASON_COLORS[season],
                  alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
    
    r, p = stats.pearsonr(v[col], v['hips_residual'])
    sl, ic, _, _, _ = stats.linregress(v[col], v['hips_residual'])
    x_fit = np.linspace(v[col].min(), v[col].max(), 50)
    ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2, alpha=0.7)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}\nn = {len(v)}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('HIPS Residual (1/Mm)' if i == 0 else '', fontsize=11)
    ax.set_title(f'HIPS Residual vs {description}', fontsize=11, fontweight='bold')
    if i == 0:
        ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('HIPS Residual (from Best-Fit HIPS–EC Line) vs AERONET Coarse-Mode Indicators',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation summary (HIPS residual vs):')
for col, xlabel, _ in test_cols:
    v = filt.dropna(subset=['hips_residual', col])
    if len(v) > 5:
        r, p = stats.pearsonr(v[col], v['hips_residual'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f'  {xlabel}: r={r:.3f}{sig}, p={p:.2e}, n={len(v)}')

---

## 3. Compression by AOD Terciles

Is HIPS "compression" (slope < 1 in HIPS EC vs FTIR EC) worse when AOD is high?

Bin by total AOD (low/mid/high) and compare:
- HIPS–EC slope
- HIPS–EC intercept
- HIPS residual distribution

If compression is worse only when AOD is high → "optical saturation/loading artifacts become important under high aerosol loading regimes."

In [ ]:
# AOD tercile split
v = filt.dropna(subset=['hips_fabs', 'ftir_ec', COL_AOD]).copy()
v = v[v['ftir_ec'] > 0]
v['hips_fabs_raw'] = v['hips_fabs'] * MAC_VALUE

aod_33 = v[COL_AOD].quantile(0.33)
aod_67 = v[COL_AOD].quantile(0.67)

v['aod_tercile'] = pd.cut(v[COL_AOD],
                           bins=[-np.inf, aod_33, aod_67, np.inf],
                           labels=['Low AOD', 'Mid AOD', 'High AOD'])

print(f'AOD tercile thresholds: {aod_33:.3f}, {aod_67:.3f}')
print(v['aod_tercile'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
tercile_colors = {'Low AOD': '#27AE60', 'Mid AOD': '#F39C12', 'High AOD': '#E74C3C'}
tercile_markers = {'Low AOD': 'o', 'Mid AOD': 's', 'High AOD': '^'}

# --- Panel 1: HIPS Fabs vs FTIR EC, colored by AOD tercile ---
ax = axes[0]
regressions_aod = {}
for tercile in ['Low AOD', 'Mid AOD', 'High AOD']:
    g = v[v['aod_tercile'] == tercile]
    ax.scatter(g['ftir_ec'], g['hips_fabs_raw'],
              color=tercile_colors[tercile], marker=tercile_markers[tercile],
              alpha=0.5, s=50, label=tercile, edgecolors='k', linewidth=0.3)
    if len(g) > 5:
        sl, ic, r, p, se = stats.linregress(g['ftir_ec'], g['hips_fabs_raw'])
        regressions_aod[tercile] = {'slope': sl, 'intercept': ic, 'r': r, 'n': len(g)}
        x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
        ax.plot(x_fit, sl * x_fit + ic, '-', color=tercile_colors[tercile], linewidth=2.5)

lims = [0, v['ftir_ec'].max() * 1.1]
ax.plot(lims, [MAC_VALUE * x for x in lims], 'k--', alpha=0.4, label=f'MAC={MAC_VALUE}')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS Fabs (1/Mm)', fontsize=12)
ax.set_title('HIPS Fabs vs FTIR EC by AOD Tercile\n(slope = MAC)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Panel 2: Slope (MAC) and intercept by AOD tercile ---
ax = axes[1]
terciles = ['Low AOD', 'Mid AOD', 'High AOD']
x_pos = np.arange(len(terciles))
slopes = [regressions_aod.get(t, {}).get('slope', np.nan) for t in terciles]
intercepts = [regressions_aod.get(t, {}).get('intercept', np.nan) for t in terciles]

ax2 = ax.twinx()
bars1 = ax.bar(x_pos - 0.15, slopes, 0.3,
               color=[tercile_colors[t] for t in terciles],
               alpha=0.8, edgecolor='k', linewidth=0.5)
bars2 = ax2.bar(x_pos + 0.15, intercepts, 0.3,
                color=[tercile_colors[t] for t in terciles],
                alpha=0.4, edgecolor='k', linewidth=0.5, hatch='//')

ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(terciles, fontsize=10)
ax.set_ylabel('Slope (MAC, m²/g)', fontsize=11)
ax2.set_ylabel('Intercept (1/Mm)', fontsize=11, color='gray')
ax.set_title('MAC & Intercept by AOD Tercile\n(compression test)', fontsize=13, fontweight='bold')

for i, (sl, ic) in enumerate(zip(slopes, intercepts)):
    if not np.isnan(sl):
        ax.text(x_pos[i] - 0.15, sl + 0.2, f'{sl:.1f}', ha='center',
                fontsize=10, fontweight='bold')
    if not np.isnan(ic):
        offset = abs(ic) * 0.15 if ic >= 0 else -abs(ic) * 0.15
        ax2.text(x_pos[i] + 0.15, ic + offset, f'{ic:.2f}', ha='center',
                fontsize=10, fontweight='bold', color='gray')
ax.grid(True, alpha=0.3, axis='y')

# --- Panel 3: HIPS residual distribution by AOD tercile ---
ax = axes[2]
v_resid = v.dropna(subset=['hips_residual'] if 'hips_residual' in v.columns else [])
if 'hips_residual' not in v.columns:
    # Compute residuals using the global fit
    v['hips_residual'] = v['hips_fabs_raw'] - (slope_fit * v['ftir_ec'] + intercept_fit)
    v_resid = v.dropna(subset=['hips_residual'])

for tercile in terciles:
    g = v_resid[v_resid['aod_tercile'] == tercile]['hips_residual']
    if len(g) > 3:
        ax.hist(g, bins=20, alpha=0.4, color=tercile_colors[tercile],
                label=f'{tercile} (n={len(g)}, std={g.std():.2f})', density=True)

ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('HIPS Residual (1/Mm)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('HIPS Residual Distribution by AOD\n(wider = more scatter)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('HIPS Compression Test: Does High AOD Make Instrument Disagreement Worse?',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n--- AOD Tercile Summary ---')
for tercile in terciles:
    r = regressions_aod.get(tercile, {})
    if r:
        g = v[v['aod_tercile'] == tercile]
        print(f'{tercile}: MAC={r["slope"]:.1f}, intercept={r["intercept"]:.3f}, '
              f'r={r["r"]:.3f}, n={r["n"]}, '
              f'AOD range=[{g[COL_AOD].min():.3f}, {g[COL_AOD].max():.3f}]')

In [ ]:
# Compare variance: Var(HIPS) vs Var(EC) by season and AOD bin
# Compression = HIPS range shrinks relative to EC range

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Variance ratio by season
ax = axes[0]
var_data = []
for season in SEASON_ORDER:
    s = filt[(filt['season'] == season) & filt['hips_fabs'].notna() & filt['ftir_ec'].notna()]
    if len(s) > 5:
        var_hips = s['hips_fabs'].var()
        var_ec = s['ftir_ec'].var()
        var_data.append({'season': season, 'var_ratio': var_hips / var_ec,
                        'std_hips': s['hips_fabs'].std(), 'std_ec': s['ftir_ec'].std(),
                        'n': len(s)})

var_df = pd.DataFrame(var_data)
bars = ax.bar(var_df['season'], var_df['var_ratio'],
              color=[SEASON_COLORS[s] for s in var_df['season']],
              alpha=0.8, edgecolor='k', linewidth=0.5)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='No compression')
for i, row in var_df.iterrows():
    ax.text(i, row['var_ratio'] + 0.02,
            f'{row["var_ratio"]:.2f}\nn={row["n"]}',
            ha='center', fontsize=10)
ax.set_ylabel('Var(HIPS EC) / Var(FTIR EC)', fontsize=12)
ax.set_title('Variance Ratio by Season\n(<1 = HIPS compressed)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 2: Variance ratio by AOD tercile
ax = axes[1]
var_data_aod = []
for tercile in ['Low AOD', 'Mid AOD', 'High AOD']:
    s = v[v['aod_tercile'] == tercile]
    if len(s) > 5:
        var_hips = s['hips_fabs'].var()
        var_ec = s['ftir_ec'].var()
        var_data_aod.append({'tercile': tercile, 'var_ratio': var_hips / var_ec,
                             'n': len(s)})

var_aod_df = pd.DataFrame(var_data_aod)
bars = ax.bar(var_aod_df['tercile'], var_aod_df['var_ratio'],
              color=[tercile_colors[t] for t in var_aod_df['tercile']],
              alpha=0.8, edgecolor='k', linewidth=0.5)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='No compression')
for i, row in var_aod_df.iterrows():
    ax.text(i, row['var_ratio'] + 0.02,
            f'{row["var_ratio"]:.2f}\nn={row["n"]}',
            ha='center', fontsize=10)
ax.set_ylabel('Var(HIPS EC) / Var(FTIR EC)', fontsize=12)
ax.set_title('Variance Ratio by AOD Tercile\n(<1 = more compression)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Signal Compression: Does HIPS Lose Dynamic Range?',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Daytime (10–14h) vs 24h Representativeness

AERONET is daytime clear-sky only. Filters integrate 24h. Addis has strong diurnal structure.

Test: does restricting aethalometer averaging to the **daytime AERONET reporting window** (10:00–14:00 local) change the agreement with:
- FTIR EC (24h filter)
- HIPS EC (24h filter)

If agreement improves → "Instrument comparisons are sensitive to diurnal representativeness."

In [ ]:
# Compute daytime-only (10-14h local) BC averages for each filter date
bc_local = bc_raw[['IR BCc']].dropna().copy()
bc_local['date'] = bc_local.index.date
bc_local['hour'] = bc_local.index.hour

# For each date, compute 24h and daytime-only averages
daily_bc = []
for date, day_data in bc_local.groupby('date'):
    full_24h = day_data['IR BCc']
    daytime = day_data[day_data['hour'].between(10, 14)]['IR BCc']  # AERONET window
    morning = day_data[day_data['hour'].between(6, 9)]['IR BCc']
    
    if len(full_24h) < 60 * 12:  # need ≥12h coverage
        continue
    
    rec = {
        'date': pd.Timestamp(date),
        'bc_24h': full_24h.mean(),
        'bc_24h_n': full_24h.notna().sum(),
    }
    if len(daytime) >= 30:  # ≥30 min in window
        rec['bc_daytime'] = daytime.mean()
        rec['bc_daytime_n'] = daytime.notna().sum()
    else:
        rec['bc_daytime'] = np.nan
        rec['bc_daytime_n'] = 0
    if len(morning) >= 30:
        rec['bc_morning'] = morning.mean()
    else:
        rec['bc_morning'] = np.nan
    
    daily_bc.append(rec)

bc_windows = pd.DataFrame(daily_bc)
bc_windows['date'] = bc_windows['date'].dt.normalize()

print(f'Days with BC window data: {len(bc_windows)}')
print(f'  With daytime (10-14h): {bc_windows["bc_daytime"].notna().sum()}')
print(f'  24h mean: {bc_windows["bc_24h"].mean():.3f} µg/m³')
print(f'  Daytime mean: {bc_windows["bc_daytime"].mean():.3f} µg/m³')
print(f'  Morning mean: {bc_windows["bc_morning"].mean():.3f} µg/m³')

In [ ]:
# Merge window-averaged BC with filter data
filt_windows = filt.copy()
filt_windows['date_norm'] = pd.to_datetime(filt_windows['date']).dt.normalize()
filt_windows = filt_windows.merge(bc_windows, left_on='date_norm',
                                   right_on='date', how='left',
                                   suffixes=('', '_win'))

n_both = filt_windows.dropna(subset=['bc_24h', 'bc_daytime', 'ftir_ec']).shape[0]
print(f'Filter dates with both 24h and daytime BC: {n_both}')

In [ ]:
# Compare: BC_24h vs EC  |  BC_daytime vs EC  |  BC_morning vs EC
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

windows = [
    ('bc_24h', '24h Average BC', '#2C3E50'),
    ('bc_daytime', 'Daytime (10-14h) BC', '#3498DB'),
    ('bc_morning', 'Morning (6-9h) BC', '#E74C3C'),
]

# Row 1: BC window vs FTIR EC
for i, (bc_col, bc_label, color) in enumerate(windows):
    ax = axes[0, i]
    v = filt_windows.dropna(subset=[bc_col, 'ftir_ec'])
    v = v[v['ftir_ec'] > 0]
    if len(v) < 5:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue
    
    for season in SEASON_ORDER:
        s = v[v['season'] == season]
        ax.scatter(s['ftir_ec'], s[bc_col], color=SEASON_COLORS[season],
                  alpha=0.5, s=40, label=season, edgecolors='k', linewidth=0.3)
    
    r, p = stats.pearsonr(v['ftir_ec'], v[bc_col])
    sl, ic, _, _, _ = stats.linregress(v['ftir_ec'], v[bc_col])
    x_fit = np.linspace(0, v['ftir_ec'].max() * 1.1, 50)
    ax.plot(x_fit, sl * x_fit + ic, '-', color=color, linewidth=2.5)
    
    lims = [0, max(v['ftir_ec'].max(), v[bc_col].max()) * 1.1]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='1:1')
    
    ax.text(0.05, 0.95, f'r = {r:.3f}\nslope = {sl:.2f}\nn = {len(v)}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel('FTIR EC (µg/m³)', fontsize=11)
    ax.set_ylabel(f'{bc_label} (µg/m³)', fontsize=11)
    ax.set_title(f'{bc_label} vs FTIR EC', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Row 2: BC window vs HIPS EC
for i, (bc_col, bc_label, color) in enumerate(windows):
    ax = axes[1, i]
    v = filt_windows.dropna(subset=[bc_col, 'hips_fabs'])
    if len(v) < 5:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue
    
    for season in SEASON_ORDER:
        s = v[v['season'] == season]
        ax.scatter(s['hips_fabs'], s[bc_col], color=SEASON_COLORS[season],
                  alpha=0.5, s=40, label=season, edgecolors='k', linewidth=0.3)
    
    r, p = stats.pearsonr(v['hips_fabs'], v[bc_col])
    sl, ic, _, _, _ = stats.linregress(v['hips_fabs'], v[bc_col])
    x_fit = np.linspace(0, v['hips_fabs'].max() * 1.1, 50)
    ax.plot(x_fit, sl * x_fit + ic, '-', color=color, linewidth=2.5)
    
    lims = [0, max(v['hips_fabs'].max(), v[bc_col].max()) * 1.1]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='1:1')
    
    ax.text(0.05, 0.95, f'r = {r:.3f}\nslope = {sl:.2f}\nn = {len(v)}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel('HIPS EC (µg/m³, MAC=10)', fontsize=11)
    ax.set_ylabel(f'{bc_label} (µg/m³)', fontsize=11)
    ax.set_title(f'{bc_label} vs HIPS EC', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Daytime vs 24h BC: Does Time-Window Restriction Improve Agreement?',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary table of agreement metrics by window
print('\n' + '=' * 80)
print('AGREEMENT METRICS BY AVERAGING WINDOW')
print('=' * 80)

for ref_col, ref_label in [('ftir_ec', 'FTIR EC'), ('hips_fabs', 'HIPS EC')]:
    print(f'\n--- vs {ref_label} ---')
    for bc_col, bc_label, _ in windows:
        v = filt_windows.dropna(subset=[bc_col, ref_col])
        if ref_col == 'ftir_ec':
            v = v[v[ref_col] > 0]
        if len(v) > 5:
            r, p = stats.pearsonr(v[ref_col], v[bc_col])
            sl, ic, _, _, _ = stats.linregress(v[ref_col], v[bc_col])
            rmse = np.sqrt(((v[bc_col] - v[ref_col])**2).mean())
            bias = (v[bc_col] - v[ref_col]).mean()
            print(f'  {bc_label:25s}: r={r:.3f}, slope={sl:.2f}, '
                  f'intercept={ic:.3f}, RMSE={rmse:.3f}, bias={bias:+.3f}, n={len(v)}')

print('\n--- Key question: does daytime-only BC agree better with FTIR EC? ---')
for bc_col, bc_label, _ in windows:
    v = filt_windows.dropna(subset=[bc_col, 'ftir_ec'])
    v = v[v['ftir_ec'] > 0]
    if len(v) > 5:
        r, _ = stats.pearsonr(v['ftir_ec'], v[bc_col])
        print(f'  {bc_label}: r with FTIR EC = {r:.3f}')

In [ ]:
# Daytime-only AERONET-matched comparison:
# Merge AERONET all-points with daytime BC for sub-daily cross-comparison

# AERONET all-points PW during 10-14h local (= 07-11h UTC) for each day
aod_all_local = aod_all.copy()
aod_all_local['hour_utc'] = aod_all_local.index.hour
aod_all_local['hour_local'] = aod_all_local['hour_utc'] + 3  # Addis = UTC+3
aod_all_local['date'] = aod_all_local.index.normalize()

# Median AERONET PW during daytime window (10-14h local = 7-11h UTC)
daytime_aeronet = aod_all_local[
    aod_all_local['hour_local'].between(10, 14)
].groupby('date').agg({
    COL_PW: 'median',
    COL_AOD: 'median',
}).rename(columns={COL_PW: 'pw_daytime', COL_AOD: 'aod_daytime'})

print(f'Days with daytime AERONET PW: {daytime_aeronet["pw_daytime"].notna().sum()}')

# Merge with filter data
filt_daytime = filt_windows.merge(
    daytime_aeronet, left_on='date_norm', right_index=True, how='left')

# Compare: MAC using daytime PW vs daily PW
v = filt_daytime.dropna(subset=['mac_hips', COL_PW, 'pw_daytime']).copy()
q01, q99 = v['mac_hips'].quantile(0.01), v['mac_hips'].quantile(0.99)
v = v[(v['mac_hips'] >= q01) & (v['mac_hips'] <= q99)]

if len(v) > 5:
    r_daily, p_daily = stats.pearsonr(v[COL_PW], v['mac_hips'])
    r_daytime, p_daytime = stats.pearsonr(v['pw_daytime'], v['mac_hips'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for i, (pw_col, pw_label, r_val) in enumerate([
        (COL_PW, 'Daily Average PW', r_daily),
        ('pw_daytime', 'Daytime (10-14h) PW', r_daytime),
    ]):
        ax = axes[i]
        for season in SEASON_ORDER:
            s = v[v['season'] == season]
            ax.scatter(s[pw_col], s['mac_hips'], color=SEASON_COLORS[season],
                      alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
        
        sl, ic, _, _, _ = stats.linregress(v[pw_col], v['mac_hips'])
        x_fit = np.linspace(v[pw_col].min(), v[pw_col].max(), 100)
        ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2.5)
        ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.5)
        
        ax.text(0.05, 0.95, f'r = {r_val:.3f}\nn = {len(v)}',
                transform=ax.transAxes, fontsize=11, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        ax.set_xlabel(pw_label + ' (cm)', fontsize=12)
        ax.set_ylabel('MAC$_{HIPS}$ (m²/g)', fontsize=12)
        ax.set_title(f'MAC vs {pw_label}', fontsize=13, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Does Daytime-Only PW Improve MAC Prediction?',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f'MAC_HIPS vs Daily PW: r = {r_daily:.3f}')
    print(f'MAC_HIPS vs Daytime PW: r = {r_daytime:.3f}')
else:
    print(f'Only {len(v)} overlapping samples — insufficient for comparison')

---

## 5. Synthesis: Three Paper-Ready Plots

The three deliverables that directly support core aims (MAC methodology + interference + why Addis is weird):

1. **MAC_HIPS vs AERONET PW** (with season color/shape)
2. **HIPS residual vs τc** (coarse-mode AOD from SDA)
3. **HIPS–EC slope/intercept split by AOD terciles**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Plot 1: MAC_HIPS vs PW ---
ax = axes[0]
v1 = filt.dropna(subset=['mac_hips', COL_PW]).copy()
q01, q99 = v1['mac_hips'].quantile(0.01), v1['mac_hips'].quantile(0.99)
v1 = v1[(v1['mac_hips'] >= q01) & (v1['mac_hips'] <= q99)]

season_markers = {'Dry': 'o', 'Belg': 's', 'Kiremt': '^'}
for season in SEASON_ORDER:
    s = v1[v1['season'] == season]
    ax.scatter(s[COL_PW], s['mac_hips'], color=SEASON_COLORS[season],
              marker=season_markers[season], alpha=0.6, s=60,
              label=season, edgecolors='k', linewidth=0.3)

r, p = stats.pearsonr(v1[COL_PW], v1['mac_hips'])
sl, ic, _, _, _ = stats.linregress(v1[COL_PW], v1['mac_hips'])
x_fit = np.linspace(v1[COL_PW].min(), v1[COL_PW].max(), 100)
ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2.5)
ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.6,
           label=f'Default MAC={MAC_VALUE}')

ax.text(0.05, 0.95, f'r = {r:.3f}, R² = {r**2:.3f}\np = {p:.2e}\nn = {len(v1)}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set_xlabel('AERONET Precipitable Water (cm)', fontsize=12)
ax.set_ylabel('MAC$_{HIPS}$ (m²/g)', fontsize=12)
ax.set_title('(A) MAC Tracks Column Moisture', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 2: HIPS residual vs coarse AOD ---
ax = axes[1]
v2 = filt.dropna(subset=['hips_residual', COL_TC])
if len(v2) > 5:
    for season in SEASON_ORDER:
        s = v2[v2['season'] == season]
        ax.scatter(s[COL_TC], s['hips_residual'], color=SEASON_COLORS[season],
                  marker=season_markers[season], alpha=0.6, s=60,
                  label=season, edgecolors='k', linewidth=0.3)
    
    r2, p2 = stats.pearsonr(v2[COL_TC], v2['hips_residual'])
    sl2, ic2, _, _, _ = stats.linregress(v2[COL_TC], v2['hips_residual'])
    x_fit2 = np.linspace(v2[COL_TC].min(), v2[COL_TC].max(), 50)
    ax.plot(x_fit2, sl2 * x_fit2 + ic2, 'k-', linewidth=2.5)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.text(0.05, 0.95, f'r = {r2:.3f}\np = {p2:.2e}\nn = {len(v2)}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_xlabel('Coarse-Mode AOD$_{500}$ (τ$_c$)', fontsize=12)
ax.set_ylabel('HIPS Residual (1/Mm)', fontsize=12)
ax.set_title('(B) HIPS Residual vs Coarse Loading', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 3: HIPS-EC by AOD tercile ---
ax = axes[2]
v3 = filt.dropna(subset=['hips_fabs', 'ftir_ec', COL_AOD]).copy()
v3 = v3[v3['ftir_ec'] > 0]
v3['hips_fabs_raw'] = v3['hips_fabs'] * MAC_VALUE
aod_33 = v3[COL_AOD].quantile(0.33)
aod_67 = v3[COL_AOD].quantile(0.67)
v3['aod_tercile'] = pd.cut(v3[COL_AOD],
                            bins=[-np.inf, aod_33, aod_67, np.inf],
                            labels=['Low AOD', 'Mid AOD', 'High AOD'])

for tercile in ['Low AOD', 'Mid AOD', 'High AOD']:
    g = v3[v3['aod_tercile'] == tercile]
    ax.scatter(g['ftir_ec'], g['hips_fabs_raw'],
              color=tercile_colors[tercile], alpha=0.5, s=50,
              label=tercile, edgecolors='k', linewidth=0.3)
    if len(g) > 5:
        sl, ic, r, _, _ = stats.linregress(g['ftir_ec'], g['hips_fabs_raw'])
        x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
        ax.plot(x_fit, sl * x_fit + ic, '-', color=tercile_colors[tercile],
                linewidth=2.5)

lims = [0, v3['ftir_ec'].max() * 1.1]
ax.plot(lims, [MAC_VALUE * x for x in lims], 'k--', alpha=0.4, label=f'MAC={MAC_VALUE}')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS Fabs (1/Mm)', fontsize=12)
ax.set_title('(C) HIPS–EC by AOD Loading Regime', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Three Paper-Ready Plots: MAC, Interference, & Compression',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print('=' * 80)
print('NOTEBOOK SUMMARY')
print('=' * 80)

print('''
1. DUAL MAC (HIPS vs AETH vs PW):
   - MAC_HIPS and MAC_AETH both computed relative to FTIR EC
   - Test: which optical method is more sensitive to column moisture?
   - Multivariate regression: MAC ~ PW + season (does season add beyond PW?)
   - Paper claim: "Observed MAC variability is associated with atmospheric
     water vapor conditions, consistent with optical-property changes affecting
     filter-based absorption (but not thermal EC)."

2. HIPS RESIDUAL vs COARSE-MODE REGIME:
   - Residual from best-fit HIPS-EC line (not raw ratio) vs τc, FMF, τc/τa
   - Tests whether "coarse/dusty optical regime" pushes HIPS off baseline
   - Framing: "measurement interference" not "sources"
   - Important caveat: coarse mode ≠ pure dust, but useful regime marker

3. COMPRESSION BY AOD TERCILES:
   - HIPS-EC slope/intercept in low/mid/high AOD bins
   - Variance ratio Var(HIPS)/Var(EC) by season and AOD
   - Tests: "optical saturation/loading artifacts under high aerosol loading"

4. DAYTIME (10-14h) vs 24h:
   - Does restricting BC averaging to AERONET reporting hours improve agreement?
   - Also: does using daytime-only PW improve MAC prediction?
   - Framing: "sampling-window comparability" (methodological, not source-based)

NOTE: No inversion data (SSA/AAOD) available for this site.
      The analysis stays purely within AOD + SDA products.
''')

# Key numbers for the paper
print('--- Key Numbers for Paper ---')
v1 = filt.dropna(subset=['mac_hips', COL_PW])
q01, q99 = v1['mac_hips'].quantile(0.01), v1['mac_hips'].quantile(0.99)
v1 = v1[(v1['mac_hips'] >= q01) & (v1['mac_hips'] <= q99)]
r_mac_pw, p_mac_pw = stats.pearsonr(v1[COL_PW], v1['mac_hips'])
print(f'MAC_HIPS vs PW: r={r_mac_pw:.3f}, R²={r_mac_pw**2:.3f}, p={p_mac_pw:.2e}, n={len(v1)}')

v2 = filt.dropna(subset=['hips_residual', COL_TC])
if len(v2) > 5:
    r_resid_tc, p_resid_tc = stats.pearsonr(v2[COL_TC], v2['hips_residual'])
    print(f'HIPS residual vs τc: r={r_resid_tc:.3f}, p={p_resid_tc:.2e}, n={len(v2)}')

print(f'\nBest-fit HIPS-EC: slope={slope_fit:.1f} m²/g (= bulk MAC), '
      f'intercept={intercept_fit:.3f}')
print(f'AERONET data: Level 2.0, daytime clear-sky only, column measurements')
print(f'Explicit in paper: AERONET is column daytime, not a surface 24h measurement.')